# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

Implements the full RAG pipeline:
- **Chunking** — sentence-boundary splitting with overlap; short/stub entries filtered
- **Embedding** — `all-MiniLM-L6-v2` dense embeddings with title prefix
- **Retrieval** — Hybrid BM25 + dense with Reciprocal Rank Fusion (RRF)
- **Generation** — Qwen2.5-0.5B-Instruct with adaptive prompting
- **Output** — saves `test_outputs.json` and `test_outputs_baseline.json`

**Files:**
- Input corpus: `Background_Corpus_Final_Combined.json`
- Input queries: `benchmark_input_only.json`
- Output: `test_outputs.json`

In [1]:
# Install dependencies (run once)
!pip install sentence-transformers transformers rank-bm25 accelerate -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import re
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from collections import Counter

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

print("All imports OK")

All imports OK


In [4]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
print("Cross-Encoder loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-Encoder loaded!


In [5]:
DATA_DIR = Path(".")

corpus_path = DATA_DIR / "background_corpus.json"
query_path  = DATA_DIR / "input_payload.json"
output_path = DATA_DIR / "output_payload.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(query_path, "r", encoding="utf-8") as f:
    query_payload = json.load(f)

queries = query_payload["queries"]

print(f"Corpus: {len(corpus)} documents")
print(f"Queries: {len(queries)}")

Corpus: 204 documents
Queries: 100


In [6]:
@dataclass
class Chunk:
    doc_id: str
    source_name: str
    title: str
    section: str
    text: str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

In [7]:
CHUNK_WORDS = 220
OVERLAP_WORDS = 40
MIN_ENTRY_CHARS = 300
MIN_CHUNK_WORDS = 25

def split_into_chunks(text: str, max_words=CHUNK_WORDS, overlap=OVERLAP_WORDS) -> List[str]:
    """Split text into overlapping sentence-boundary chunks."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        if current and current_len + len(words) > max_words:
            chunks.append(" ".join(current))

            # keep overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                wlen = len(s.split())
                if overlap_len + wlen > overlap:
                    break
                overlap_sents.insert(0, s)
                overlap_len += wlen

            current = overlap_sents
            current_len = overlap_len

        current.append(sent)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks


def make_section_chunks(doc_id: str, source_name: str, title: str, section_name: str, text: str) -> List[Chunk]:
    text = text.strip()
    if len(text.split()) < MIN_CHUNK_WORDS:
        return []

    if len(text.split()) <= CHUNK_WORDS:
        return [Chunk(doc_id=doc_id, title=title, section=section_name, text=text, source_name=source_name)] # Added source_name here

    parts = split_into_chunks(text)
    return [
        Chunk(
            doc_id=doc_id,
            source_name=source_name,
            title=title,
            section=f"{section_name} (Part {i+1})",
            text=part
        )
        for i, part in enumerate(parts)
        if len(part.split()) >= MIN_CHUNK_WORDS
    ]


def chunk_documents(corpus_docs: list) -> List[Chunk]:
    chunks = []

    for doc in corpus_docs:
        doc_id = doc.get("doc_id", "")
        source_name = doc.get("source_name", doc.get("title", "Unknown Source"))
        title = doc.get("title", "")
        sections = doc.get("sections", [])
        full_text = doc.get("full_text", "").strip()

        # skip scaffold/meta docs
        if "scaffold" in doc_id or "scope" in doc_id:
            continue

        # skip extremely short docs
        if not sections and len(full_text) < MIN_ENTRY_CHARS:
            continue

        if sections:
            for sec in sections:
                heading = sec.get("heading", "Section")
                text = " ".join(sec.get("content", [])).strip()
                # Corrected the arguments passed to make_section_chunks
                chunks.extend(make_section_chunks(doc_id, source_name, title, heading, text))
        else:
            chunks.extend(make_section_chunks(doc_id, source_name, title, "Full text", full_text))

    return chunks


chunks = chunk_documents(corpus)

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length (words): {sum(len(c.text.split()) for c in chunks) // len(chunks)}")

chunk_id_map = {id(chunk): idx for idx, chunk in enumerate(chunks)}

wiki_chunks = [c for c in chunks if c.doc_id.startswith("wiki")]
print(f"Wiki chunks in index: {len(wiki_chunks)}")
print(f"Chunks under 30 words: {sum(1 for c in chunks if len(c.text.split()) < 30)}")

Total chunks: 446
Avg chunk length (words): 129
Wiki chunks in index: 116
Chunks under 30 words: 17


In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded:", embedding_model.get_sentence_embedding_dimension(), "dims")

chunk_texts = [
    f"passage: {c.title}. {c.section}. {c.text}"
    for c in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

chunk_embeddings = chunk_embeddings.astype("float32")
chunk_embeddings_norm = chunk_embeddings / (np.linalg.norm(chunk_embeddings, axis=1, keepdims=True) + 1e-12)

print(f"Embeddings shape: {chunk_embeddings_norm.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: 384 dims


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (446, 384)


In [9]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("english")

def tokenize(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)?", text)
    return [stemmer.stem(w) for w in words if len(w) > 2]

tokenized_chunks = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index built over", len(tokenized_chunks), "chunks")

BM25 index built over 446 chunks


In [10]:
DENSE_CANDIDATES = 50
BM25_CANDIDATES = 50

def dense_retrieve(query: str, top_k: int = DENSE_CANDIDATES) -> List[RetrievalResult]:
    q_text = f"query: {query}"
    q_emb = embedding_model.encode([q_text], convert_to_numpy=True)[0].astype("float32")
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-12)

    sims = chunk_embeddings_norm @ q_emb
    top_idx = np.argsort(sims)[::-1][:top_k]

    return [RetrievalResult(chunk=chunks[i], score=float(sims[i])) for i in top_idx]


def bm25_retrieve(query: str, top_k: int = BM25_CANDIDATES) -> List[RetrievalResult]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [RetrievalResult(chunk=chunks[i], score=float(scores[i])) for i in top_idx]

In [11]:
RRF_K = 60
TOP_K = 5
MAX_PER_DOC = 5

def hybrid_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    # 1. Cast a wide net: Get top 20 from both
    dense_res = dense_retrieve(query, top_k=40)
    bm25_res  = bm25_retrieve(query, top_k=40)

    # 2. Merge unique chunks using a dictionary
    unique_candidates = {}
    for res in dense_res + bm25_res:
        key = res.chunk.text
        if key not in unique_candidates:
            unique_candidates[key] = res.chunk

    candidates_list = list(unique_candidates.values())

    # 3. Re-rank with Cross-Encoder
    cross_inp = [[query, chunk.text] for chunk in candidates_list]
    cross_scores = reranker.predict(cross_inp)

    # 4. Sort by the Cross-Encoder score
    scored_candidates = []
    for chunk, score in zip(candidates_list, cross_scores):
        scored_candidates.append(RetrievalResult(chunk=chunk, score=float(score)))

    scored_candidates = sorted(scored_candidates, key=lambda x: x.score, reverse=True)

    # 5. Apply Document Diversity (Max per doc)
    selected = []
    doc_counts = {}

    for res in scored_candidates:
        did = res.chunk.doc_id
        if doc_counts.get(did, 0) >= MAX_PER_DOC:
            continue

        selected.append(res)
        doc_counts[did] = doc_counts.get(did, 0) + 1

        if len(selected) == top_k:
            break

    return selected

def baseline_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    return dense_retrieve(query, top_k=top_k)

In [12]:
test_query = "What is biryani and which region is it originally associated with?"
test_results = hybrid_retrieve(test_query)

print(f"Top {len(test_results)} chunks retrieved:\n")
for i, r in enumerate(test_results, 1):
    print(f"{i}. [{r.chunk.doc_id} | {r.chunk.section}] score={r.score:.4f}")
    print(f"   {r.chunk.text[:160]}...")
    print()

Top 5 chunks retrieved:

1. [crossregional_dish_biryani_variants | Full text (Part 1)] score=6.6243
   Biryani is the most celebrated and widely eaten rice dish across South Asia, existing in dozens of distinct regional variants that reflect the history, culture,...

2. [wiki_biryani | Introduction] score=6.4074
   Biryani is a rice dish made with rice, spices, and usually meat, although vegetarian versions also exist. It is one of the best-known dishes associated with Sou...

3. [india_dish_biryani | Full text (Part 1)] score=5.9933
   Biryani is a mixed rice dish originating in South Asia, traditionally made with long-grain basmati rice, meat (chicken, goat, beef, lamb, or seafood), aromatic ...

4. [india_dish_biryani | Full text (Part 2)] score=4.7476
   Regional varieties include Hyderabadi biryani (made with goat meat and saffron, considered a culinary emblem of the city), Lucknawi Awadhi biryani, Kolkata biry...

5. [bangladesh_dish_kacchi_biryani | Full text (Part 2)] score=3.9

In [13]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cpu


In [14]:
def split_sentences(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 20]

def sentence_overlap_score(query: str, sentence: str) -> float:
    q_tokens = set(re.findall(r"[a-z0-9]+", query.lower()))
    s_tokens = set(re.findall(r"[a-z0-9]+", sentence.lower()))
    if not q_tokens:
        return 0.0

    overlap = len(q_tokens & s_tokens) / len(q_tokens)

    q_lower = query.lower()
    s_lower = sentence.lower()

    if any(w in q_lower for w in ["ingredient", "ingredients", "include", "contain"]):
        if any(w in s_lower for w in ["include", "includes", "contains", "made with", "ingredients"]):
            overlap += 0.15

    if any(w in q_lower for w in ["what is", "what are"]):
        if any(w in s_lower for w in [" is ", " are ", "refers to", "dish made from"]):
            overlap += 0.10

    if "served with" in q_lower and "served with" in s_lower:
        overlap += 0.20

    if "introduced" in q_lower and "introduced" in s_lower:
        overlap += 0.20

    return overlap

def extract_relevant_sentences(query: str, retrieved: List[RetrievalResult], max_sentences: int = 8) -> List[str]:
    scored = []

    for r in retrieved:
        for sent in split_sentences(r.chunk.text):
            score = sentence_overlap_score(query, sent)
            scored.append((score, r.chunk.title, r.chunk.section, sent))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    selected = []
    seen = set()

    for score, title, section, sent in scored:
        key = sent.strip().lower()
        if key in seen:
            continue
        seen.add(key)
        selected.append(f"[Source: {title} | {section}] {sent}")
        if len(selected) == max_sentences:
            break

    return selected

In [15]:
MAX_CONTEXT_CHARS = 1800

def build_prompt(query: str, retrieved: List[RetrievalResult]) -> List[dict]:
    evidence_texts = []
    char_count = 0
    for r in retrieved:
        chunk_text = f"[Source: {r.chunk.title} | {r.chunk.section}] {r.chunk.text}"
        if char_count + len(chunk_text) > MAX_CONTEXT_CHARS:
            break
        evidence_texts.append(chunk_text)
        char_count += len(chunk_text)

    evidence = "\n\n".join(evidence_texts)
    # ---------------------

    q_lower = query.lower()

    # (Keep your excellent adaptive prompting exactly as it was)
    if any(w in q_lower for w in ["ingredient", "ingredients", "include", "contain"]):
        answer_instruction = (
            "Answer with only the ingredients explicitly mentioned in the evidence. "
            "Do not add any ingredient that is not stated. "
            "Write 1 to 2 complete sentences."
        )
    elif "served with" in q_lower:
        answer_instruction = (
            "Answer only with what the dish is served with, using the evidence. "
            "Write 1 to 2 complete sentences."
        )
    elif any(w in q_lower for w in ["what is", "what are"]):
        answer_instruction = (
            "Give a short direct definition using only the evidence. "
            "Do not add preparation details unless asked. "
            "Write 1 to 2 complete sentences."
        )
    elif "introduced" in q_lower:
        answer_instruction = (
            "Answer only with the techniques or items explicitly stated as introduced in the evidence. "
            "Do not add unrelated historical details. "
            "Write 1 to 2 complete sentences."
        )
    else:
        answer_instruction = (
            "Answer directly and concisely using only the evidence below. "
            "Do not use conversational filler like 'Based on the evidence'."
            "Write 1 to 2 complete sentences."
        )

    return [
        {
            "role": "system",
            "content": (
                "You are an expert South Asian cuisine assistant. "
                "You answer questions directly, concisely, and exclusively using the provided evidence. "
                "Do not guess. Do not add outside knowledge. "
            )
        },
        {
            "role": "user",
            "content": (
                "Evidence:\n[Source: wiki_dish | Intro] Samosa is a fried or baked pastry with a savory filling, including ingredients such as spiced potatoes, onions, and peas.\n\n"
                "Question: What are the main ingredients of a samosa?\n\n"
                "Answer with only the ingredients explicitly mentioned in the evidence. Write 1 to 2 complete sentences."
            )
        },
        {
            "role": "assistant",
            "content": "The main ingredients of a samosa explicitly mentioned in the evidence are spiced potatoes, onions, and peas."
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n{evidence}\n\n"
                f"Question: {query}\n\n"
                f"{answer_instruction}"
            )
        }
    ]

In [16]:
def postprocess(text: str) -> str:
    text = text.strip()

    for prefix in ["Answer:", "Response:", "A:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    text = re.sub(r"\s+", " ", text).strip()

    # remove trailing incomplete fragment only if clearly cut off
    if text and text[-1] not in ".!?":
        parts = re.split(r'(?<=[.!?])\s+', text)
        if len(parts) > 1:
            text = " ".join(parts[:-1]).strip()

    return text


def generate(messages: list) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.05,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return postprocess(tokenizer.decode(new_ids, skip_special_tokens=True))

In [17]:
sample_query = queries[0]["query"]
sample_results = hybrid_retrieve(sample_query)
sample_messages = build_prompt(sample_query, sample_results)
sample_answer = generate(sample_messages)

print("QUERY: ", sample_query)
print("ANSWER:", sample_answer)
print()
print("Retrieved docs:")
for r in sample_results:
    print(f"  - {r.chunk.doc_id} | {r.chunk.section}")

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUERY:  What is biryani and which culinary tradition is it most closely associated with?
ANSWER: Biryani is a rice dish that originated in South Asia and is most closely associated with the Persian cuisine tradition.

Retrieved docs:
  - wiki_biryani | Introduction
  - india_dish_biryani | Full text (Part 1)
  - india_technique_dum_cooking | Full text (Part 2)
  - wiki_pilaf | Full text (Part 2)
  - bangladesh_dish_kacchi_biryani | Full text (Part 1)


In [19]:
results = []

for i, item in enumerate(queries):
    query_id = item["query_id"]
    query = item["query"]

    retrieved = hybrid_retrieve(query, top_k=TOP_K)
    messages = build_prompt(query, retrieved)
    response = generate(messages)

    results.append({
        "query_id": query_id,
        "query": query,
        "response": response,
        "retrieved_context": [
            {
                "doc_id": r.chunk.doc_id,
                "text": r.chunk.text
            }
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(queries)} done")

print(f"\nGenerated {len(results)} answers")

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done

Generated 100 answers


In [20]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({"results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to {output_path}")

Saved 100 results to output_payload.json


In [21]:
baseline_results = []

for i, item in enumerate(queries):
    query_id = item["query_id"]
    query = item["query"]

    retrieved = baseline_retrieve(query, top_k=TOP_K)
    messages = build_prompt(query, retrieved)
    response = generate(messages)

    baseline_results.append({
        "query_id": query_id,
        "query": query,
        "response": response,
        "retrieved_context": [
            {"doc_id": r.chunk.doc_id, "text": r.chunk.text}
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(queries)} done")

baseline_path = DATA_DIR / "test_outputs_baseline.json"
with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump({"results": baseline_results}, f, indent=2, ensure_ascii=False)

print(f"Baseline saved to {baseline_path}")

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Baseline saved to test_outputs_baseline.json


In [22]:
import gradio as gr

def chat_with_rag(message, history):
    """
    Wrapper function to connect your existing RAG pipeline to the Gradio ChatInterface.
    """
    # 1. Retrieve relevant chunks using your hybrid search
    retrieved_chunks = hybrid_retrieve(message, top_k=5)

    # 2. Build the messages using your existing function
    messages = build_prompt(message, retrieved_chunks)

    # 3. Generate the answer
    answer = generate(messages)

    # 4. Format the retrieved context nicely to show it off during the demo
    context_str = "\n\n".join([
    # Slice the text, not just the section title
    f"*[{r.chunk.title} | {r.chunk.section}]* (Score: {r.score:.4f})\n{r.chunk.text[:150]}..."
    for r in retrieved_chunks
])

    # 5. Return the combined output
    return f"{answer}\n\n---\n### 🔍 Retrieved Context\n{context_str}"

# Create the Chat Interface
demo = gr.ChatInterface(
    fn=chat_with_rag,
    title="🍲 Culinary Assistant (South Asian Cuisine)",
    description="Ask me anything about South Asian cuisine! My answers are grounded in a curated RAG pipeline."
)

# Launch the UI directly below this cell
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ffc5f27b8a800722ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
